<a href="https://colab.research.google.com/github/LoneWolf2026/Neural-Network-for-Nuclear-Binding-Energy-Prediction-/blob/main/Nuclear_Binding_Energy_NeuralNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#Omit uncertainties for now. Add them later. Focus on getting the Neural net trained first
#1. Optimize Gradient Descent functions
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [5]:
data_df_2020 = pd.read_csv('/content/drive/MyDrive/AME_Dataset_2020.csv',header =None)
data_df_2020 = data_df_2020.drop([0, 1]).reset_index(drop=True) #removes neutron and proton from first two rows
data_df_2020 = data_df_2020.drop(columns=[0,4]).reset_index(drop=True)
data_df_2020.columns = range(data_df_2020.columns.size)
data_df_2020.head()

,0,1,2,3,4,5,6,7,8
0,1,1,2,13135.722895,0.000015,1112.28310,0.00020,14101.777844,0.000015
1,2,1,3,14949.810900,0.000080,2827.26540,0.00030,16049.281320,0.000080
2,1,2,3,14931.218880,0.000060,2572.68044,0.00015,16029.321970,0.000060
3,0,3,3,28667.000000,2000.000000,-2267.00000,667.00000,30775.000000,2147.000000
4,3,1,4,24621.129000,100.000000,1720.44910,25.00000,26431.867000,107.354000


In [6]:
og_data = data_df_2020.copy()


In [10]:
input = np.array(data_df_2020.iloc[:,[0,1,2,3,7]].values) #Neutrons, Protons, Mass Number, Mass Excess, and Atomic Mass are take as inputs
output = np.array(data_df_2020.iloc[:,[5]].values) #Binding Energy is our output

print(input)
print(output)


[[1.00000000e+00 1.00000000e+00 2.00000000e+00 1.31357229e+04
  1.41017778e+04]
 [2.00000000e+00 1.00000000e+00 3.00000000e+00 1.49498109e+04
  1.60492813e+04]
 [1.00000000e+00 2.00000000e+00 3.00000000e+00 1.49312189e+04
  1.60293220e+04]
 ...
 [1.77000000e+02 1.17000000e+02 2.94000000e+02 1.96397000e+05
  2.10840000e+05]
 [1.76000000e+02 1.18000000e+02 2.94000000e+02 1.99320000e+05
  2.13979000e+05]
 [1.77000000e+02 1.18000000e+02 2.95000000e+02 2.01369000e+05
  2.16178000e+05]]
[[1112.2831 ]
 [2827.2654 ]
 [2572.68044]
 ...
 [7092.     ]
 [7079.     ]
 [7076.     ]]


In [17]:
input_train,input_test,output_train,output_test = train_test_split(input,output,test_size = 0.3)

#Standardize input and output training data for simpler training
scaler_input = StandardScaler()
scaler_output = StandardScaler()
#must have two different scaler functions since input and output have different column dimensions

input_train = scaler_input.fit_transform(input_train)
output_train = scaler_output.fit_transform(output_train)

input_test = scaler_input.transform(input_test)
output_test = scaler_output.transform(output_test)

input_test,input_val,output_test,output_val = train_test_split(input_test,output_test,test_size = 0.5)

In [20]:
class dataset(Dataset):
  def __init__(self,input,output):
    self.input = torch.tensor(input, dtype = torch.float32).to(device)
    self.output = torch.tensor(output, dtype = torch.float32).to(device)

  def __len__(self):
    return len(self.input)

  def __getitem__(self,index):
    return self.input[index], self.output[index]

In [21]:
training_data = dataset(input_train,output_train)
validation_data = dataset(input_val,output_val)
testing_data = dataset(input_test,output_test)

In [22]:
train_dataloader = DataLoader(training_data,batch_size = 4,shuffle = True)
val_dataloader = DataLoader(validation_data,batch_size = 4,shuffle = True)
test_dataloader = DataLoader(testing_data,batch_size = 4,shuffle = True)

In [24]:
for i,b in train_dataloader:
  print(i)
  print("===========")
  print(b)
  break

tensor([[-0.0692,  0.0818, -0.0103, -1.0402,  0.5044],
        [-0.1153, -0.5279, -0.2787, -0.2133,  0.6338],
        [-0.5298, -0.5279, -0.5331, -1.0852,  0.4974],
        [-0.1383,  0.0100, -0.0809, -1.0897,  0.4967]], device='cuda:0')
tensor([[0.5119],
        [0.1862],
        [0.8461],
        [0.5689]], device='cuda:0')


In [25]:

class BindingE_NeuralNet(nn.Module):
  def __init__(self):
    super(BindingE_NeuralNet,self).__init__()

    self.layer1 = nn.Linear(input_train.shape[1],64)
    self.layer2 = nn.Linear(64,32)
    self.OutLayer = nn.Linear(32,1) #Output is Binding Energy
    self.Relu = nn.ReLU()

  def forward(self,X):

    X = self.Relu(self.layer1(X))
    X = self.Relu(self.layer2(X))
    X = self.OutLayer(X)

    #Return to this Model and refine architecture

    return X


BE_NeuralNet = BindingE_NeuralNet().to('cuda')

In [26]:
summary(BE_NeuralNet,(input.shape[1],))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 64]             384
              ReLU-2                   [-1, 64]               0
            Linear-3                   [-1, 32]           2,080
              ReLU-4                   [-1, 32]               0
            Linear-5                    [-1, 1]              33
Total params: 2,497
Trainable params: 2,497
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.01
Estimated Total Size (MB): 0.01
----------------------------------------------------------------


In [28]:
criterion = nn.MSELoss() #Mean Square Error Loss function
optimizer = Adam(BE_NeuralNet.parameters(), lr = 0.001)

In [29]:
for data in train_dataloader:
  inputs, labels = data
  print(inputs)
  print(labels)
  break

tensor([[ 0.4605,  0.3328,  0.4136, -0.6318,  0.5683],
        [ 0.6677,  0.9425,  0.7810,  0.2067,  0.6995],
        [ 1.6580,  1.5880,  1.6430,  1.9468, -1.5909],
        [-0.2074, -0.1334, -0.1798, -1.1511,  0.4871]], device='cuda:0')
tensor([[ 0.2215],
        [-0.2085],
        [-0.7156],
        [ 0.6614]], device='cuda:0')


In [30]:
total_loss_train_plot = []
total_loss_validation_plot = []
total_acc_train_plot = []
total_acc_validation_plot = []

epochs = 10
for epoch in range(epochs):
  total_acc_train = 0
  total_loss_train = 0
  total_acc_validation = 0
  total_loss_validation = 0

  for data in train_dataloader:
    inputs, labels = data

    prediction = BE_NeuralNet(inputs)

    batch_loss = criterion(prediction,labels)
    total_loss_train += batch_loss.item()

    acc = ((prediction).round() == labels).sum().item()

    total_acc_train += acc

    batch_loss.backward()
    optimizer.step()
    optimizer.zero_grad()

  with torch.no_grad():
    for data in val_dataloader:
      inputs, labels = data

      prediction = BE_NeuralNet(inputs)
      batch_loss = criterion(prediction,labels)

      total_loss_validation += batch_loss.item()
      acc = ((prediction).round() == labels).sum().item()

      total_acc_validation += acc

  total_loss_train_plot.append(round(total_loss_train/1000,4))
  total_loss_validation_plot.append(round(total_loss_validation/1000,4))

  total_acc_train_plot.append(round(total_acc_train/training_data.__len__()*100,4))
  total_acc_validation_plot.append(round(total_acc_validation/validation_data.__len__()*100,4))

  print(f'''Epoch: {epoch+1} Train_Loss: {round(total_loss_train/1000,4)} Train_Accuracy: {round(total_acc_train/training_data.__len__()*100,4)}
        Validation Loss: {round(total_loss_validation/1000,4)} Validation_Accuracy: {round(total_acc_validation/validation_data.__len__()*100,4)}''')
  print("=="*25)

Epoch: 1 Train_Loss: 0.2451 Train_Accuracy: 0.0
        Validation Loss: 0.0251 Validation_Accuracy: 0.0
Epoch: 2 Train_Loss: 0.1876 Train_Accuracy: 0.0
        Validation Loss: 0.0205 Validation_Accuracy: 0.0
Epoch: 3 Train_Loss: 0.1688 Train_Accuracy: 0.0
        Validation Loss: 0.0166 Validation_Accuracy: 0.0
Epoch: 4 Train_Loss: 0.1441 Train_Accuracy: 0.0
        Validation Loss: 0.0126 Validation_Accuracy: 0.0
Epoch: 5 Train_Loss: 0.1289 Train_Accuracy: 0.0
        Validation Loss: 0.0102 Validation_Accuracy: 0.0
Epoch: 6 Train_Loss: 0.1142 Train_Accuracy: 0.0
        Validation Loss: 0.013 Validation_Accuracy: 0.0
Epoch: 7 Train_Loss: 0.0964 Train_Accuracy: 0.0
        Validation Loss: 0.008 Validation_Accuracy: 0.0
Epoch: 8 Train_Loss: 0.1039 Train_Accuracy: 0.0
        Validation Loss: 0.0068 Validation_Accuracy: 0.0
Epoch: 9 Train_Loss: 0.0932 Train_Accuracy: 0.0
        Validation Loss: 0.0066 Validation_Accuracy: 0.0
Epoch: 10 Train_Loss: 0.0905 Train_Accuracy: 0.0
        